# Восстановление пунктуации (мультимодальное, late fusion / вариант A)

Модель преобразует **текст без пунктуации → текст с пунктуацией**, опираясь *одновременно* на:
* сам текст;
* акустические признаки из звука — **смысловые паузы**, длительности слов, темп, **F0** (интонация) и энергию.

Восстанавливаются: **запятые, точки, многоточия, вопросительные и восклицательные знаки**, а также **абзацы (красные строки)** и **капитализация**.

Вход — текст *или* звук; выход — текст с пунктуацией. Модель спроектирована как **отдельный блок SpeechToText-пайплайна**: выход Whisper (слова + тайм-коды) подаётся в `PunctuationRestorer`.

### Архитектуры
1. **BiLSTM** — baseline (нижняя планка).
2. **Transformer-энкодер с нуля** — baseline-трансформер (отделяет вклад архитектуры от предобучения).
3. **RuBERT** (`DeepPavlov/rubert-base-cased`) и **rubert-tiny2** (`cointegrated/rubert-tiny2`) — предобученные.

Все модели двухпоточные: текстовый энкодер ⊕ MLP-энкодер акустики → 3 головы (`punct` / `para` / `cap`).

### Структура проекта
```
punctuation_restoration/
├── punctuation_pipeline.ipynb   <- этот ноутбук (только импорт + запуск)
├── requirements.txt
└── modules/
    ├── config.py          метки, признаки, гиперпараметры
    ├── data.py            FLEURS, forced alignment, акустика, разметка
    ├── tokenizer.py       словарь для baseline
    ├── dataset.py         Dataset + collate (baseline / pretrained)
    ├── train.py           общий цикл обучения
    ├── evaluate.py        P/R/F1 по классам
    ├── inference.py        PunctuationRestorer + STTPunctuationPipeline
    └── models/
        ├── heads.py            акустический энкодер + 3 головы (общие)
        ├── lstm_model.py
        ├── transformer_model.py
        └── pretrained_model.py
```

## 0. Установка зависимостей

In [1]:
# !pip install -r requirements.txt
# Forced aligner (для акустики на FLEURS):
# !pip install git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git

# На Windows для чтения аудио FLEURS:
import os
os.environ.setdefault("DATASETS_AUDIO_BACKEND", "soundfile")

'soundfile'

## 1. Импорт модулей

In [8]:
import numpy as np
import torch
from functools import partial
from torch.utils.data import DataLoader

from modules.config import get_config, PRETRAINED_PRESETS, PUNCT_LABELS, PARA_LABELS, CAP_LABELS, ACOUSTIC_FEATURES
from modules.data import build_examples
from modules.dataset import WordVocab, BaselineDataset, PretrainedDataset, baseline_collate, pretrained_collate
from modules.evaluate import evaluate, pretty_report
from modules.inference import PunctuationRestorer, STTPunctuationPipeline
from modules.models.pretrained_model import load_hf_tokenizer
from modules.train import train_model, set_seed
from modules.models.__init__ import build_model

cfg = get_config()
set_seed(cfg.train.seed)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg.train.device = DEVICE
print("device:", DEVICE)
print("пунктуация:", PUNCT_LABELS)
print("абзац:", PARA_LABELS, "| капитализация:", CAP_LABELS)
print("акустические признаки:", ACOUSTIC_FEATURES)

ModuleNotFoundError: No module named 'modules.models.config'

## 2. Данные: FLEURS (ru_ru)

`build_examples` грузит FLEURS, парсит транскрипции в метки трёх голов и (если установлен forced-aligner) считает акустические признаки на каждое слово.

* `use_alignment=True` — полный мультимодальный режим (нужен `ctc-forced-aligner` + аудио).
* `use_alignment=False` — text-only (акустика = нули), удобно для быстрой проверки.
* Если FLEURS/сеть недоступны — вернутся демонстрационные примеры, чтобы ноутбук исполнялся.

Подберите `limit` под железо (полный train FLEURS-ru ≈ 2.5k примеров).

In [ ]:
USE_ALIGNMENT = True   # False -> быстрый text-only прогон без аудио
LIMIT_TRAIN = 800      # None = весь сплит
LIMIT_VAL   = 200

train_examples = build_examples(cfg.data, split="train",      limit=LIMIT_TRAIN, use_alignment=USE_ALIGNMENT)
val_examples   = build_examples(cfg.data, split="validation", limit=LIMIT_VAL,   use_alignment=USE_ALIGNMENT)

print(f"train: {len(train_examples)} | val: {len(val_examples)}")
ex = train_examples[0]
print("слова:", ex.words[:12])
print("есть акустика:", ex.has_acoustic, "| форма признаков:", ex.acoustic.shape)

### (опц.) Пересчёт статистик нормализации акустики на train
Грубые `ACOUSTIC_NORM` в `config.py` можно заменить реальными mean/std корпуса — обычно даёт небольшой прирост.

In [ ]:
# from modules.data import compute_acoustic_stats
# stats = compute_acoustic_stats(train_examples)
# import pprint; pprint.pprint(stats)
# затем перенесите значения в config.ACOUSTIC_NORM и пересоберите примеры

---
## 3. Baseline №1 — BiLSTM

Самая лёгкая модель, обучается с нуля. Текстовый словарь строится по train-корпусу.

In [ ]:
vocab = WordVocab.build(train_examples, min_freq=1, max_size=50000)
print("размер словаря:", len(vocab))

train_ds = BaselineDataset(train_examples, vocab, max_len=cfg.train.max_len)
val_ds   = BaselineDataset(val_examples,   vocab, max_len=cfg.train.max_len)

collate = partial(baseline_collate, pad_id=vocab.pad_id)
train_loader = DataLoader(train_ds, batch_size=cfg.train.batch_size, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(val_ds,   batch_size=cfg.train.batch_size, shuffle=False, collate_fn=collate)

lstm = build_model("lstm", vocab_size=len(vocab), pad_id=vocab.pad_id, use_acoustic=True)
print("параметров:", sum(p.numel() for p in lstm.parameters()))

In [ ]:
cfg.train.epochs = 5
cfg.train.lr = 1e-3
lstm = train_model(lstm, train_loader, cfg.train, val_loader=val_loader, eval_fn=evaluate)

In [ ]:
lstm_metrics = evaluate(lstm, val_loader, torch.device(DEVICE))
print(pretty_report(lstm_metrics))

---
## 4. Baseline №2 — Transformer-энкодер с нуля

Та же подготовка данных (словарь/лоадеры переиспользуются), но архитектура — self-attention, обучаемая без предобучения.

In [ ]:
transformer = build_model("transformer", vocab_size=len(vocab), pad_id=vocab.pad_id, use_acoustic=True)
print("параметров:", sum(p.numel() for p in transformer.parameters()))

cfg.train.epochs = 5
cfg.train.lr = 5e-4
transformer = train_model(transformer, train_loader, cfg.train, val_loader=val_loader, eval_fn=evaluate)
tr_metrics = evaluate(transformer, val_loader, torch.device(DEVICE))
print(pretty_report(tr_metrics))

---
## 5. Предобученные — RuBERT-base и rubert-tiny2

Subword-токенизация HF; метка слова ставится на первый субтокен. Дообучаем энкодер (малый lr) + новые головы (больший lr) — это уже зашито в `train_model(is_pretrained=True)`.

Выберите пресет: `rubert-base` (сильнее) или `rubert-tiny2` (быстрее/легче).

In [ ]:
PRESET = "rubert-base"   # или "rubert-tiny2"
model_name = PRETRAINED_PRESETS[PRESET]
print("модель:", model_name)

hf_tok = load_hf_tokenizer(model_name)
ptr_train_ds = PretrainedDataset(train_examples, hf_tok, max_len=cfg.train.max_len)
ptr_val_ds   = PretrainedDataset(val_examples,   hf_tok, max_len=cfg.train.max_len)

ptr_collate = partial(pretrained_collate, pad_id=hf_tok.pad_token_id or 0)
ptr_train_loader = DataLoader(ptr_train_ds, batch_size=cfg.train.batch_size, shuffle=True,  collate_fn=ptr_collate)
ptr_val_loader   = DataLoader(ptr_val_ds,   batch_size=cfg.train.batch_size, shuffle=False, collate_fn=ptr_collate)

pretrained = build_model("pretrained", model_name=model_name, use_acoustic=True)

In [ ]:
cfg.train.epochs = 3   # предобученным хватает меньше эпох
pretrained = train_model(pretrained, ptr_train_loader, cfg.train,
                         val_loader=ptr_val_loader, is_pretrained=True, eval_fn=evaluate)
ptr_metrics = evaluate(pretrained, ptr_val_loader, torch.device(DEVICE))
print(pretty_report(ptr_metrics))

### Сравнение моделей (macro-F1 пунктуации на валидации)

In [ ]:
import pandas as pd
rows = []
for name, m in [("BiLSTM", lstm_metrics), ("Transformer", tr_metrics), (PRESET, ptr_metrics)]:
    rows.append({
        "модель": name,
        "punct macro-F1": round(m.get("punct_f1_macro", 0), 3),
        "para macro-F1":  round(m.get("para_f1_macro", 0), 3),
        "cap macro-F1":   round(m.get("cap_f1_macro", 0), 3),
    })
pd.DataFrame(rows)

---
## 6. Инференс: текст без пунктуации → текст с пунктуацией

`PunctuationRestorer` оборачивает любую обученную модель. Два режима:
* `restore(text)` — только текст;
* `restore_from_words(words, word_ts, audio, sr)` — мультимодальный (использует паузы/F0).

In [ ]:
# baseline
restorer = PunctuationRestorer(lstm, kind="lstm", vocab=vocab, device=DEVICE)
# предобученная:
# restorer = PunctuationRestorer(pretrained, kind="pretrained", hf_tokenizer=hf_tok, device=DEVICE)

print(restorer.restore("привет как дела я давно тебя не видел"))
print(restorer.restore("сегодня хорошая погода может прогуляемся по набережной"))

## 7. Встраивание в SpeechToText-пайплайн

Whisper отдаёт слова и их тайм-коды. Из тайм-кодов считаются те же акустические признаки, что и при обучении — паузы определяют точки/запятые/абзацы, F0 помогает с `?`/`!`.

`STTPunctuationPipeline` принимает выход STT в виде словаря:
* `{"text": ...}` → text-only путь;
* `{"words": [...], "word_timestamps": [{word,start,end}], "audio": np.ndarray}` → мультимодальный.

Ниже — демонстрация с искусственными тайм-кодами (большая пауза = граница предложения). В реальном пайплайне `stt_output` формирует ваша Whisper-модель.

In [ ]:
pipeline = STTPunctuationPipeline(restorer)

# --- имитация выхода Whisper: слова + тайм-коды ---
words = "привет как дела я давно тебя не видел".split()
t, word_ts = 0.0, []
for w in words:
    dur = 0.3
    word_ts.append({"word": w, "start": round(t, 2), "end": round(t + dur, 2)})
    gap = 0.6 if w in ("дела", "видел") else 0.05   # длинные паузы -> границы
    t += dur + gap

stt_output = {"words": words, "word_timestamps": word_ts, "audio": None}
print("С учётом пауз :", pipeline(stt_output))
print("Только текст  :", pipeline({"text": " ".join(words)}))

### Как подключить реальный Whisper
```python
import whisper
asr = whisper.load_model("large-v3")
res = asr.transcribe("audio.wav", language="ru", word_timestamps=True)

words, word_ts = [], []
for seg in res["segments"]:
    for w in seg["words"]:
        token = w["word"].strip()
        words.append(token)
        word_ts.append({"word": token, "start": w["start"], "end": w["end"]})

import soundfile as sf
audio, sr = sf.read("audio.wav")           # для F0/энергии
stt_output = {"words": words, "word_timestamps": word_ts, "audio": audio}
final_text = STTPunctuationPipeline(restorer)(stt_output, sr=sr)
```
Так модель пунктуации становится вторым звеном пайплайна `звук → текст без пунктуации → текст с пунктуацией`.

## 8. Сохранение / загрузка
```python
torch.save(lstm.state_dict(), "lstm_punct.pt"); vocab.save("vocab.json")
torch.save(pretrained.state_dict(), "rubert_punct.pt")
```
Загрузка: пересоздать модель через `build_model(...)`, затем `load_state_dict`. Для baseline дополнительно `WordVocab.load("vocab.json")`.